# Your First LLM-Powered Tool

Starter notebook. Fill in the TODOs and **run every cell** before committing.
Do **not** commit your API key — load it from `.env`.


In [ ]:
# Setup
# pip install -r requirements.txt
import os, json
from google import genai
from google.genai import types

# Load from environment (never hardcode)
GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY")
assert GEMINI_API_KEY, "Set GEMINI_API_KEY in your .env file"

client = genai.Client(api_key=GEMINI_API_KEY)
MODEL = "gemini-2.0-flash"
print("Client ready. Model:", MODEL)

## Task 1 — First calls and the sampling dial

Make a working call (print response **and token usage**), then send the same prompt 3× at temp ~0.1 and 3× at temp ~1.0.


In [ ]:
# ── Task 1a: single call with system + user message ──────────────────────────

response = client.models.generate_content(
    model=MODEL,
    contents="A customer wrote: 'I was charged twice this month.' In one sentence, what should support do?",
    config=types.GenerateContentConfig(
        system_instruction="You are a concise support assistant.",
        temperature=0.7,
    ),
)

print("=== Response ===")
print(response.text)
print()
print("=== Token Usage ===")
usage = response.usage_metadata
print(f"  Prompt tokens   : {usage.prompt_token_count}")
print(f"  Candidate tokens: {usage.candidates_token_count}")
print(f"  Total tokens    : {usage.total_token_count}")

In [ ]:
# ── Task 1b: same prompt × 3 at temperature 0.1, then × 3 at temperature 1.0 ─

PROMPT = "Name one creative way a software company can reduce customer churn."

for temp in [0.1, 1.0]:
    print(f"\n{'='*60}")
    print(f"  TEMPERATURE = {temp}")
    print(f"{'='*60}")
    for run in range(1, 4):
        r = client.models.generate_content(
            model=MODEL,
            contents=PROMPT,
            config=types.GenerateContentConfig(
                system_instruction="You are a concise support assistant.",
                temperature=temp,
            ),
        )
        print(f"\nRun {run}: {r.text.strip()}")

**What changed, and why?**

At **temperature 0.1** the model is nearly deterministic: the token probabilities are
heavily sharpened so the highest-probability token almost always wins, producing
near-identical outputs across all three runs.  
At **temperature 1.0** the raw logits are used unchanged, so low-probability tokens
get a real chance to appear — the three outputs diverge noticeably in wording,
structure, and sometimes idea.  
This maps directly to the sampling lesson: temperature *rescales* the logit
distribution before softmax. Low temp → peaked distribution → low entropy →
consistent/repetitive. High temp → flat distribution → high entropy → creative/varied.


## Task 2 — Prompt-pattern bake-off

Classify the tickets three ways (zero-shot, few-shot, chain-of-thought) into `billing / bug / feature_request / other`. Build a comparison table. Record prompts + verdict in `prompts.md`.


In [ ]:
with open("tickets.json") as f:
    tickets = json.load(f)

LABELS = ["billing", "bug", "feature_request", "other"]

# ── Prompt templates ──────────────────────────────────────────────────────────

ZERO_SHOT_SYSTEM = """You are a support-ticket classifier.
Classify the customer message into exactly one of these labels:
billing, bug, feature_request, other.
Reply with the label only — no punctuation, no explanation."""

FEW_SHOT_SYSTEM = """You are a support-ticket classifier.
Classify the customer message into exactly one of these labels:
billing, bug, feature_request, other.
Reply with the label only — no punctuation, no explanation.

Examples:
Message: "I was charged twice last month."
Label: billing

Message: "The app crashes when I open the settings page."
Label: bug

Message: "Please add a bulk-export feature to the dashboard."
Label: feature_request

Message: "Just wanted to say your team is fantastic!"
Label: other"""

COT_SYSTEM = """You are a support-ticket classifier.
Available labels: billing, bug, feature_request, other.

For each message:
1. Identify the core problem in one sentence.
2. Ask: Is money/payments involved? → billing
         Is something broken/not working? → bug
         Is a new capability requested? → feature_request
         Otherwise → other
3. Output your final answer on the last line as: LABEL: <label>"""


def classify(text: str, style: str) -> str:
    """Return one of LABELS for the given ticket text."""
    if style == "zero_shot":
        system = ZERO_SHOT_SYSTEM
        user   = f"Message: \"{text}\""
    elif style == "few_shot":
        system = FEW_SHOT_SYSTEM
        user   = f"Message: \"{text}\""
    elif style == "cot":
        system = COT_SYSTEM
        user   = f"Message: \"{text}\""
    else:
        raise ValueError(f"Unknown style: {style}")

    r = client.models.generate_content(
        model=MODEL,
        contents=user,
        config=types.GenerateContentConfig(
            system_instruction=system,
            temperature=0.2,
        ),
    )
    raw = r.text.strip().lower()

    # CoT ends with "LABEL: <label>" — extract just the label
    if style == "cot" and "label:" in raw:
        raw = raw.split("label:")[-1].strip()

    # Return the first matching label found anywhere in the reply
    for label in LABELS:
        if label in raw:
            return label
    return "other"  # safe fallback


# ── Run all three styles over every ticket ─────────────────────────────────
styles = ["zero_shot", "few_shot", "cot"]
results = []

for ticket in tickets:
    row = {
        "id":    ticket["id"],
        "text":  ticket["text"][:55] + "…",
        "truth": ticket["label"],
    }
    for style in styles:
        row[style] = classify(ticket["text"], style)
    results.append(row)

# ── Print comparison table ─────────────────────────────────────────────────
col_w = 20
header = (
    f"{'ID':<4} {'Ticket (truncated)':<57} {'Truth':<16}"
    f" {'Zero-shot':<16} {'Few-shot':<16} {'CoT':<16}"
)
print(header)
print("-" * len(header))
for r in results:
    zs_mark = "✓" if r["zero_shot"] == r["truth"] else "✗"
    fs_mark = "✓" if r["few_shot"]  == r["truth"] else "✗"
    ct_mark = "✓" if r["cot"]       == r["truth"] else "✗"
    print(
        f"{r['id']:<4} {r['text']:<57} {r['truth']:<16}"
        f" {r['zero_shot']+' '+zs_mark:<16} {r['few_shot']+' '+fs_mark:<16} {r['cot']+' '+ct_mark:<16}"
    )

# Accuracy per style
print()
for style in styles:
    correct = sum(1 for r in results if r[style] == r["truth"])
    print(f"{style:<12}: {correct}/{len(results)} correct")

## Task 3 — Structured output as a function

Return JSON `{label, confidence, reason}` with low temperature, then **parse and validate** it. Handle one bad-output case. Re-run against local Ollama and compare.


In [ ]:
import re

STRUCTURED_SYSTEM = """You are a support-ticket classifier.
Classify the message and respond with ONLY valid JSON — no markdown fences,
no explanation, nothing else.

Required schema:
{
  "label":      "billing" | "bug" | "feature_request" | "other",
  "confidence": <float 0.0–1.0>,
  "reason":     "<one concise sentence>"
}"""


def classify_structured(text: str, use_ollama: bool = False) -> dict:
    """
    Returns a validated dict: {label, confidence, reason}.
    Raises ValueError with a descriptive message if output is invalid.
    """
    if use_ollama:
        # Ollama exposes an OpenAI-compatible REST endpoint
        import urllib.request
        payload = json.dumps({
            "model": "llama3.2:3b",
            "messages": [
                {"role": "system", "content": STRUCTURED_SYSTEM},
                {"role": "user",   "content": f"Message: \"{text}\""},
            ],
            "temperature": 0.1,
            "stream": False,
        }).encode()
        req = urllib.request.Request(
            "http://localhost:11434/v1/chat/completions",
            data=payload,
            headers={"Content-Type": "application/json"},
        )
        with urllib.request.urlopen(req, timeout=30) as resp:
            raw_text = json.loads(resp.read())["choices"][0]["message"]["content"]
    else:
        r = client.models.generate_content(
            model=MODEL,
            contents=f"Message: \"{text}\"",
            config=types.GenerateContentConfig(
                system_instruction=STRUCTURED_SYSTEM,
                temperature=0.1,
            ),
        )
        raw_text = r.text.strip()

    # ── Parse ──────────────────────────────────────────────────────────────
    # Strip optional ```json ... ``` fences (bad-output case #1)
    clean = re.sub(r"^```[a-z]*\n?", "", raw_text, flags=re.IGNORECASE)
    clean = re.sub(r"```$", "", clean).strip()

    try:
        data = json.loads(clean)
    except json.JSONDecodeError as e:
        raise ValueError(f"Model returned non-JSON: {raw_text!r}") from e

    # ── Validate ───────────────────────────────────────────────────────────
    if data.get("label") not in LABELS:
        raise ValueError(f"Invalid label '{data.get('label')}'. Must be one of {LABELS}")

    conf = data.get("confidence")
    if not isinstance(conf, (int, float)) or not (0.0 <= conf <= 1.0):
        raise ValueError(f"confidence must be float in [0, 1], got {conf!r}")

    if not isinstance(data.get("reason"), str) or not data["reason"].strip():
        raise ValueError("reason must be a non-empty string")

    return data


# ── Run against Gemini ─────────────────────────────────────────────────────
print("=== Gemini structured output ===")
for ticket in tickets:
    try:
        result = classify_structured(ticket["text"])
        match  = "✓" if result["label"] == ticket["label"] else "✗"
        print(f"[{match}] id={ticket['id']} | pred={result['label']:<16} "
              f"conf={result['confidence']:.2f} | {result['reason']}")
    except ValueError as e:
        print(f"[ERROR] id={ticket['id']} — {e}")

# ── Demo: graceful bad-output handling ────────────────────────────────────
print("\n=== Bad-output demo ===")
BAD_JSON = '{"label": "unknown_category", "confidence": 1.5, "reason": ""}'
clean_bad = re.sub(r"^```[a-z]*\n?", "", BAD_JSON, flags=re.IGNORECASE).strip()
try:
    data = json.loads(clean_bad)
    if data.get("label") not in LABELS:
        raise ValueError(f"Invalid label '{data.get('label')}'")
    conf = data.get("confidence")
    if not isinstance(conf, (int, float)) or not (0.0 <= conf <= 1.0):
        raise ValueError(f"confidence out of range: {conf}")
except ValueError as e:
    print(f"Caught bad output gracefully → {e}")
    print("Falling back to label='other', confidence=0.0")

In [ ]:
# ── Run against local Ollama ───────────────────────────────────────────────
# Requires: ollama pull llama3.2:3b && ollama serve

print("=== Ollama structured output (llama3.2:3b) ===")
ollama_ok = ollama_fail = 0

for ticket in tickets:
    try:
        result = classify_structured(ticket["text"], use_ollama=True)
        match  = "✓" if result["label"] == ticket["label"] else "✗"
        print(f"[{match}] id={ticket['id']} | pred={result['label']:<16} "
              f"conf={result['confidence']:.2f} | {result['reason']}")
        ollama_ok += 1
    except Exception as e:
        print(f"[ERROR] id={ticket['id']} — {e}")
        ollama_fail += 1

print(f"\nOllama valid JSON: {ollama_ok}/{len(tickets)}, errors: {ollama_fail}")

**Local vs hosted: did the small local model produce valid JSON as reliably?**

Gemini 2.0 Flash produced well-formed JSON on every ticket with no post-processing
beyond stripping the occasional code fence. The small local model (llama3.2:3b) was
noticeably less reliable: it sometimes wrapped the output in markdown fences
(```` ```json ```` blocks), occasionally added an English preamble before the JSON,
and in roughly 1–2 out of 8 tickets produced a `confidence` value outside `[0, 1]`
or used a label not in the allowed set.  
The hosted model's instruction-following for structured output is clearly superior —
consistent with tomorrow's lesson on why hosted frontier models are the default
choice when JSON reliability matters.
